<a href="https://colab.research.google.com/github/ncrowder/maven/blob/main/maven_drill_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Data here: https://mavenanalytics.io/data-drills/booking-breakdown
df = pd.read_csv('hotel_bookings.csv',
                 parse_dates=['booking_date','cancel_date','checkin_date','checkout_date'],
                 dtype = {'is_canceled': "boolean"},
                 ).set_index('booking_id')

In [ ]:
df.cancel_date.isna().sum()

np.int64(28538)

In [ ]:
df = df[~df['is_canceled']]

In [ ]:
df

,booking_date,cancel_date,checkin_date,checkout_date,is_canceled
booking_id,,,,,
1,2014-03-18,NaT,2016-02-25,2016-03-24,False
2,2014-04-18,NaT,2015-10-02,2015-10-11,False
3,2014-04-30,NaT,2015-08-03,2015-08-10,False
4,2014-04-30,NaT,2015-08-03,2015-08-10,False
5,2014-04-30,NaT,2015-08-03,2015-08-10,False
...,...,...,...,...,...
39357,2017-08-30,NaT,2017-08-30,2017-08-31,False
39358,2017-08-30,NaT,2017-08-30,2017-08-31,False
39359,2017-08-30,NaT,2017-08-30,2017-09-01,False


In [ ]:
df1 = df.drop(columns = ['cancel_date','is_canceled','booking_date'])

In [ ]:
df1

,checkin_date,checkout_date
booking_id,,
1,2016-02-25,2016-03-24
2,2015-10-02,2015-10-11
3,2015-08-03,2015-08-10
4,2015-08-03,2015-08-10
5,2015-08-03,2015-08-10
...,...,...
39357,2017-08-30,2017-08-31
39358,2017-08-30,2017-08-31
39359,2017-08-30,2017-09-01


In [ ]:
df1.checkin_date.min()

Timestamp('2015-07-01 00:00:00')

In [ ]:
df1.checkout_date.max()-pd.Timedelta(days=1)

Timestamp('2017-09-13 00:00:00')

In [ ]:
df1_daterange = pd.date_range(start=df1.checkin_date.min(),end=df1.checkout_date.max()-pd.Timedelta(days=1))

In [ ]:
df1_daterange

DatetimeIndex(['2015-07-01', '2015-07-02', '2015-07-03', '2015-07-04',
               '2015-07-05', '2015-07-06', '2015-07-07', '2015-07-08',
               '2015-07-09', '2015-07-10',
               ...
               '2017-09-04', '2017-09-05', '2017-09-06', '2017-09-07',
               '2017-09-08', '2017-09-09', '2017-09-10', '2017-09-11',
               '2017-09-12', '2017-09-13'],
              dtype='datetime64[ns]', length=806, freq='D')

In [ ]:
df1_dates = df1_daterange.values
checkin = df['checkin_date'].values
checkout = df['checkout_date'].values

In [ ]:
mask = (df1_dates[:, None] >= checkin) & (df1_dates[:, None] < checkout)

In [ ]:
mask

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [ ]:
counts = mask.sum(axis=1)
answer = pd.DataFrame({'count':counts},index=df1_dates)
answer

,count
2015-07-01,35
2015-07-02,63
2015-07-03,80
2015-07-04,107
2015-07-05,121
...,...
2017-09-09,7
2017-09-10,3
2017-09-11,3
2017-09-12,2


In [ ]:
answer['occupancy'] = (answer['count']/200*100).round(2)
answer

,count,occupancy
2015-07-01,35,17.5
2015-07-02,63,31.5
2015-07-03,80,40.0
2015-07-04,107,53.5
2015-07-05,121,60.5
...,...,...
2017-09-09,7,3.5
2017-09-10,3,1.5
2017-09-11,3,1.5
2017-09-12,2,1.0


In [ ]:
answer_month = answer.groupby(answer.index.to_period('M')).mean()
answer_month

,count,occupancy
2015-07,159.483871,79.741935
2015-08,179.258065,89.629032
2015-09,180.933333,90.466667
2015-10,155.774194,77.887097
2015-11,109.833333,54.916667
2015-12,86.516129,43.258065
2016-01,68.258065,34.129032
2016-02,104.413793,52.206897
2016-03,139.322581,69.661290
2016-04,155.066667,77.533333


In [ ]:
booking = df.booking_date.dt.month.value_counts()
checkin = df.checkin_date.dt.month.value_counts()
length = ((df['checkout_date'] - df['checkin_date']).dt.days
    .groupby(df['checkin_date'].dt.month)
    .mean()
    )
cancellations = df.cancel_date.dt.month.value_counts()
data = {'booking':booking, 'checkin':checkin, 'cancellation':cancellations, 'length':length}
df_by_month_totals = pd.DataFrame(data)

In [ ]:
df_by_month_totals

,booking,checkin,cancellation,length
1.0,4263,1837,NaN,2.742515
2.0,3299,2288,NaN,3.057255
3.0,2644,2545,NaN,3.749312
4.0,1903,2529,NaN,3.811783
5.0,1809,2495,NaN,4.296192
6.0,1755,2010,NaN,5.394527
7.0,1893,3088,NaN,5.262953
8.0,1814,3219,NaN,5.225225
9.0,1921,2088,NaN,5.246648
10.0,3021,2530,NaN,3.846245


In [ ]:
booking = (
    df.groupby([df.booking_date.dt.year, df.booking_date.dt.month])
      .size()
      .unstack()
      .mean()
      .rename('booking')
)
checkin = (
    df.groupby([df.checkin_date.dt.year, df.checkin_date.dt.month])
      .size()
      .unstack()
      .mean()
      .rename('checkin')
)
length = ((df['checkout_date'] - df['checkin_date']).dt.days
    .groupby(df['checkin_date'].dt.month)
    .mean()
    )
cancellations = (
    df.groupby([df.cancel_date.dt.year, df.cancel_date.dt.month])
      .size()
      .unstack()
      .mean()
      .rename('checkin')
)
df_by_month_avg = pd.DataFrame({'booking':booking,'checkin':checkin,'length':length,'cancellations':cancellations})
df_by_month_avg

,booking,checkin,length,cancellations
1.0,1421.000000,918.500000,2.742515,NaN
2.0,1099.666667,1144.000000,3.057255,NaN
3.0,661.000000,1272.500000,3.749312,NaN
4.0,475.750000,1264.500000,3.811783,NaN
5.0,603.000000,1247.500000,4.296192,NaN
6.0,438.750000,1005.000000,5.394527,NaN
7.0,473.250000,1029.333333,5.262953,NaN
8.0,453.500000,1073.000000,5.225225,NaN
9.0,640.333333,1044.000000,5.246648,NaN
10.0,1007.000000,1265.000000,3.846245,NaN


In [ ]:
df = df.assign(
    booking_month=df['booking_date'].dt.to_period('M'),
    checkin_month=df['checkin_date'].dt.to_period('M'),
    cancel_month=df['cancel_date'].dt.to_period('M'),
    length=(df['checkout_date'] - df['checkin_date']).dt.days
)

booking = df['booking_month'].value_counts().sort_index()
checkin = df['checkin_month'].value_counts().sort_index()
cancellations = df['cancel_month'].value_counts().sort_index()

length = df.groupby('checkin_month')['length'].mean()

df_by_year_month = pd.DataFrame({
    'booking': booking,
    'checkin': checkin,
    'cancellation': cancellations,
    'length': length
})

In [ ]:
df_by_year_month

,booking,checkin,cancellation,length
2014-03,1,NaN,NaN,NaN
2014-04,4,NaN,NaN,NaN
2014-06,1,NaN,NaN,NaN
2014-07,2,NaN,NaN,NaN
2014-08,4,NaN,NaN,NaN
2014-09,12,NaN,NaN,NaN
2014-10,10,NaN,NaN,NaN
2014-11,37,NaN,NaN,NaN
2015-01,408,NaN,NaN,NaN
2015-02,210,NaN,NaN,NaN


In [ ]:
df.head()

,booking_id,booking_date,cancel_date,checkin_date,checkout_date,is_canceled,booking_month,checkin_month,cancel_month,length
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,False,2014-03,2016-02,NaT,28
1,2,2014-04-18,NaT,2015-10-02,2015-10-11,False,2014-04,2015-10,NaT,9
2,3,2014-04-30,NaT,2015-08-03,2015-08-10,False,2014-04,2015-08,NaT,7
3,4,2014-04-30,NaT,2015-08-03,2015-08-10,False,2014-04,2015-08,NaT,7
4,5,2014-04-30,NaT,2015-08-03,2015-08-10,False,2014-04,2015-08,NaT,7


In [ ]:
start = df['checkin_date'].min()
end = df['checkin_date'].max()

df_dates = pd.DataFrame(index=pd.date_range(start=start, end=end))
df_dates.index.name = 'date'

In [ ]:
df_dates.reset_index(inplace=True)
df_dates['count']=0
df_dates

,date,count
0,2015-07-01,0
1,2015-07-02,0
2,2015-07-03,0
3,2015-07-04,0
4,2015-07-05,0
...,...,...
788,2017-08-27,0
789,2017-08-28,0
790,2017-08-29,0
791,2017-08-30,0


In [ ]:
df

,booking_id,booking_date,cancel_date,checkin_date,checkout_date,is_canceled,booking_month,checkin_month,cancel_month,length
0,1,2014-03-18,NaT,2016-02-25,2016-03-24,False,2014-03,2016-02,NaT,28
1,2,2014-04-18,NaT,2015-10-02,2015-10-11,False,2014-04,2015-10,NaT,9
2,3,2014-04-30,NaT,2015-08-03,2015-08-10,False,2014-04,2015-08,NaT,7
3,4,2014-04-30,NaT,2015-08-03,2015-08-10,False,2014-04,2015-08,NaT,7
4,5,2014-04-30,NaT,2015-08-03,2015-08-10,False,2014-04,2015-08,NaT,7
...,...,...,...,...,...,...,...,...,...,...
39356,39357,2017-08-30,NaT,2017-08-30,2017-08-31,False,2017-08,2017-08,NaT,1
39357,39358,2017-08-30,NaT,2017-08-30,2017-08-31,False,2017-08,2017-08,NaT,1
39358,39359,2017-08-30,NaT,2017-08-30,2017-09-01,False,2017-08,2017-08,NaT,2
39359,39360,2017-08-30,NaT,2017-08-31,2017-09-02,False,2017-08,2017-08,NaT,2


In [ ]:
#df = df_dates.join(df)

In [ ]:
df_dates

,date,count
0,2015-07-01,0
1,2015-07-02,0
2,2015-07-03,0
3,2015-07-04,0
4,2015-07-05,0
...,...,...
788,2017-08-27,0
789,2017-08-28,0
790,2017-08-29,0
791,2017-08-30,0


In [ ]:
def booking_count(row):
  rowdate = row['date']
  mask = (rowdate >= df.checkin_date) * (rowdate < df.checkout_date)
  row['count'] = mask.sum()
  return row['count']

In [ ]:
df_dates['count'] = df_dates.apply(booking_count,axis=1)

In [ ]:
df_dates

,date,count
0,2015-07-01,35
1,2015-07-02,63
2,2015-07-03,80
3,2015-07-04,107
4,2015-07-05,121
...,...,...
788,2017-08-27,179
789,2017-08-28,174
790,2017-08-29,173
791,2017-08-30,174


In [ ]:
df_dates['occupancy'] = (df_dates['count']/200*100).round(2)

In [ ]:
df_dates

,date,count,occupancy
0,2015-07-01,35,17.5
1,2015-07-02,63,31.5
2,2015-07-03,80,40.0
3,2015-07-04,107,53.5
4,2015-07-05,121,60.5
...,...,...,...
788,2017-08-27,179,89.5
789,2017-08-28,174,87.0
790,2017-08-29,173,86.5
791,2017-08-30,174,87.0


In [ ]:
df_dates.set_index('date',inplace=True)

In [ ]:
df_dates.groupby(df_dates.index.to_period('M')).mean()

,count,occupancy
date,,
2015-07,159.483871,79.741935
2015-08,179.258065,89.629032
2015-09,180.933333,90.466667
2015-10,155.774194,77.887097
2015-11,109.833333,54.916667
2015-12,86.516129,43.258065
2016-01,68.258065,34.129032
2016-02,104.413793,52.206897
2016-03,139.322581,69.661290
